In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import rasterio
from rasterstats import zonal_stats
import os



In [ ]:
raster_folder = "C:/Users/kbonsu/Desktop/AICC Conference/ECOSTRESS_data_processing/data/summer_ndvi_ndbi_dem_2154"

# Load grid
grid = gpd.read_file("C:/Users/kbonsu/Desktop/AICC Conference/idf_200_grid.shp")


In [ ]:
grid_proj = grid.copy()  # already in same CRS
grid_proj = grid_proj[grid_proj.is_valid]
grid_proj['geometry'] = grid_proj.buffer(0)

grid_ids = grid_proj['idcar_200m'].values


In [ ]:
dem_raster = os.path.join(
    raster_folder,
    "UrbanPredictors_Summer_2018_2154.tif"
)

print("Extracting DEM once...")

dem_stats = zonal_stats(
    grid_proj,
    dem_raster,
    stats=['mean'],
    band=3,
    geojson_out=False
)

dem_values = [s['mean'] for s in dem_stats]


In [ ]:
all_years = []

for year in range(2018, 2026):

    raster_path = os.path.join(
        raster_folder,
        f"UrbanPredictors_Summer_{year}_2154.tif"
    )

    print(f"Processing {year}...")

    with rasterio.open(raster_path) as src:

        # NDVI (band 1)
        ndvi_stats = zonal_stats(
            grid_proj,
            src.read(1),
            affine=src.transform,
            stats=['mean'],
            geojson_out=False
        )

        # NDBI (band 2)
        ndbi_stats = zonal_stats(
            grid_proj,
            src.read(2),
            affine=src.transform,
            stats=['mean'],
            geojson_out=False
        )

    temp = pd.DataFrame({
        'grid_id': grid_ids,
        'year': year,
        'ndvi': [s['mean'] for s in ndvi_stats],
        'ndbi': [s['mean'] for s in ndbi_stats],
        'dem': dem_values   # reuse precomputed DEM
    })

    all_years.append(temp)

In [ ]:
ndvi_panel = pd.concat(all_years, ignore_index=True)

print("Done.")
print(ndvi_panel.head())

In [ ]:
ndvi_panel.shape

In [ ]:
dist = pd.read_csv("C:/Users/kbonsu/Desktop/AICC Conference/idf_distance_to_water.csv")

dist.head()

In [ ]:
dist.shape

In [ ]:
dist = dist.rename(columns={'idcar_200m': 'grid_id'})
dist['grid_id'] = dist['grid_id'].astype(str)
ndvi_panel['grid_id'] = ndvi_panel['grid_id'].astype(str)

In [ ]:
ndvi_panel = ndvi_panel.merge(
    dist[['grid_id', 'DistWater', 'DistWater_log']],
    on='grid_id',
    how='left'
)


In [ ]:
ndvi_panel.shape

In [ ]:
ndvi_panel[['DistWater','DistWater_log']].isna().mean()


In [ ]:
grid = grid.rename(columns={'idcar_200m': 'grid_id'})
grid['grid_id'] = grid['grid_id'].astype(str)

In [ ]:
# Avoid division by zero
grid['ind'] = grid['ind'].replace(0, np.nan)
grid['men'] = grid['men'].replace(0, np.nan)

# -----------------------
# Economic Structure
# -----------------------

grid['mean_income'] = grid['ind_snv'] / grid['ind']

grid['poverty_rate'] = grid['men_pauv'] / grid['men']

# -----------------------
# Population Density
# 200m cell = 0.04 km²
# -----------------------

grid['pop_density'] = grid['ind'] / 0.04

# -----------------------
# Housing Structure
# -----------------------

grid['share_collective'] = grid['men_coll'] / grid['men']
grid['share_house'] = grid['men_mais'] / grid['men']

# -----------------------
# Social Housing Share
# -----------------------

grid['total_dwellings'] = (
    grid['log_av45'] +
    grid['log_45_70'] +
    grid['log_70_90'] +
    grid['log_ap90']
)

grid['total_dwellings'] = grid['total_dwellings'].replace(0, np.nan)

grid['social_share'] = grid['log_soc'] / grid['total_dwellings']

# -----------------------
# Vulnerability
# -----------------------

grid['elderly_share'] = (
    grid['ind_80p']
) / grid['ind']

grid['child_share'] = (
    grid['ind_0_3'] +
    grid['ind_4_5']
) / grid['ind']


In [ ]:
grid_socio = grid[[
    'grid_id',
    'mean_income',
    'poverty_rate',
    'pop_density',
    'share_collective',
    'social_share',
    'elderly_share',
    'child_share'
]].copy()


In [ ]:
ndvi_panel['grid_id'] = ndvi_panel['grid_id'].astype(str)

ndvi_panel = ndvi_panel.merge(
    grid_socio,
    on='grid_id',
    how='left'
)
